# SIGPA-GA · Notebook 02 — Preparación y validación (Semana 21)

**DATOS DE DEMOSTRACIÓN.** Mismo origen que el notebook 01:
`exportar_dataset` hacia `ML_EXPORT_DIR`, **fuera del repositorio**.
Los números de este cuaderno comprueban el flujo; **no son
resultados de la tesis**.

## Qué hace
1. **Limpieza** con la tabla de exclusiones (`ml.limpieza`).
2. **Conjuntos** de predictores y su umbral de faltantes.
3. **Pipeline** de imputación/transformación sin fuga (`ml.preparacion`).
4. **Validación agrupada** por productor, semilla fija 42.
5. **Línea base**: la media del entrenamiento, la barra a batir.

## Qué NO hace
- **No entrena modelos**: solo prepara datos y mide la línea base.

## Rutas

La carpeta de exportación se lee del `.env` de la raíz (`ML_EXPORT_DIR`).

In [ ]:
from pathlib import Path
import json
import os
import sys

# Raiz del repositorio (funciona desde notebooks/ o desde la raiz)
_candidatas = [Path.cwd(), Path.cwd().parent]
REPOSITORIO = next(
    (p for p in _candidatas if (p / "backend").is_dir() and (p / "notebooks").is_dir()),
    Path.cwd().parent,
)
sys.path.insert(0, str(REPOSITORIO / "backend"))


def valor_entorno(nombre, por_defecto):
    """Lee el .env de la raiz; si no, el entorno; si no, el defecto."""
    archivo = REPOSITORIO / ".env"
    if archivo.exists():
        for linea in archivo.read_text(encoding="utf-8").splitlines():
            if linea.startswith(nombre + "="):
                return linea.split("=", 1)[1].strip().strip("'\"")
    return os.environ.get(nombre, por_defecto)


# Los datos viven en ML_EXPORT_DIR (fuera de Git), nunca en el repositorio
RUTA_EXPORT = Path(valor_entorno("ML_EXPORT_DIR", r"D:\MLDatasetSigpa"))
print("Carpeta de exportacion:", RUTA_EXPORT)

## Carga del dataset

`ml.carga.cargar` rechaza columnas o tipos raros antes de seguir.

In [ ]:
import pandas as pd

from ml import catalogo, linea_base, preparacion, validacion
from ml.carga import cargar
from ml.limpieza import limpiar

csvs = sorted(RUTA_EXPORT.glob("dataset_*.csv"))
if not csvs:
    raise SystemExit(
        "No hay dataset en %s. Ejecuta: manage.py exportar_dataset "
        "--usuario admin --origen-datos demostracion" % RUTA_EXPORT
    )
ruta_csv = csvs[-1]
datos = cargar(ruta_csv)
print("%s: %d filas x %d columnas" % (ruta_csv.name, datos.shape[0], datos.shape[1]))

## 1. Limpieza con la tabla de exclusiones

Solo se excluyen filas imposibles o sin objetivo. Lo inusual NO se
excluye. El resumen trae conteos, nunca el contenido de las filas.

In [ ]:
limpio, resumen = limpiar(datos)
print("Excluidas:", resumen["filas_excluidas"], "de", resumen["filas_entrada"])
for motivo, n in resumen["por_motivo"].items():
    print("  %s: %d" % (motivo, n))
print("Quedan:", resumen["filas_salida"], "filas limpias")

## 2. Conjunto 1 (Plan) y su pipeline

La imputación por mediana y el escalado se aprenden **solo** con el
entrenamiento de cada pliegue: el pipeline se ajusta dentro del
corte, nunca con todo el dataset.

In [ ]:
X, y, pipeline, fuera = preparacion.preparar(limpio, catalogo.CONJUNTO_PLAN)
print("Predictores (Conjunto 1):", list(X.columns))
print("Fuera por faltantes:", fuera or "ninguna")
print("Objetivo:", y.name, "| filas:", len(y))

## 3. Validación agrupada por productor

Un productor **no** aparece en entrenamiento y en prueba a la vez:
sus parcelas se parecen y el modelo las memorizaría. La semilla 42
hace el corte reproducible.

In [ ]:
n_grupos = int(limpio["productor_id"].nunique())
if n_grupos < 2:
    print("Hacen falta al menos 2 productores en el dataset para validar por grupos.")
else:
    pliegues = validacion.particiones(limpio, n_pliegues=min(5, n_grupos))
    for i, (train, test) in enumerate(pliegues, 1):
        grupos_train = set(limpio["productor_id"].iloc[train])
        grupos_test = set(limpio["productor_id"].iloc[test])
        assert grupos_train.isdisjoint(grupos_test), "productor compartido"
        print("Pliegue %d: entrena %d filas (%d productores) | prueba %d filas (%d productores)" % (
            i, len(train), len(grupos_train), len(test), len(grupos_test)))

## 4. Línea base: la barra a batir

Predice la media del entrenamiento de **cada pliegue** y se evalúa en
kg/ha. Cualquier modelo del Sprint 13 tiene que ganarle en todos los
pliegues.

In [ ]:
if n_grupos >= 2:
    objetivos = limpio["rendimiento_kg_ha"]
    filas = []
    for i, (train, test) in enumerate(pliegues, 1):
        media = linea_base.predecir_media(objetivos.iloc[train])
        m = linea_base.evaluar(objetivos.iloc[test], [media] * len(test))
        filas.append({"pliegue": i,
                      "media_entrenamiento": round(media, 1),
                      "mae": round(m["mae"], 1),
                      "rmse": round(m["rmse"], 1),
                      "filas_prueba": m["filas"]})
    pd.DataFrame(filas)

## Cómo reproducir

- Semilla fija **42** (`ml.exportacion.SEMILLA`): mismos pliegues siempre.
- Los datos salen de `ML_EXPORT_DIR`; dentro de Git no entra ningún CSV.
- Los números de arriba son de **demostración**: en el Sprint 13 los
  modelos tienen que ganar a la línea base en CADA pliegue.